# Similarity Evaluation

This tutorial compares three learned measures relevant to human visual judgments. You will construct representational dissimilarity matrices (RDMs), change the aspect judged by TPIPS, and examine disagreement between metrics.

Prerequisites: basic Python, `.[vision,notebooks]`, and pretrained model downloads. CUDA is recommended for TPIPS's 8B backbone. The stimuli are controlled image transformations constructed locally; **the evaluated metrics are real pretrained models**. This notebook makes no paid generation calls and is independent of the generation tour.

LPIPS was evaluated on patch-level perceptual judgments; DreamSim targets more holistic mid-level judgments; TPIPS explicitly conditions similarity on language. None is a universal measure of human perception. These distinctions are supported by [the primary papers](../docs/literature.md).

In [ ]:
%matplotlib inline
from pathlib import Path
import os
from dotenv import load_dotenv
ROOT = Path.cwd() if (Path.cwd() / "synthart").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env.local")
import numpy as np
import matplotlib.pyplot as plt
from synthart import ImageGenerator, Similarity, FlowMapGenerator, generate_similar
from synthart.images import release_memory
from synthart.plotting import gallery, plot_spaces
from synthart.experiment import cached_image
OUT = ROOT / "outputs" / "notebook-tour"
OUT.mkdir(parents=True, exist_ok=True)


## Controlled Stimuli

We vary color, geometry, and spatial arrangement separately. These simple stimuli make the manipulation visible, but they are far from the natural and generated image distributions used to train the metrics. Treat the results as a diagnostic demonstration, not a cognitive benchmark.

In [ ]:
from PIL import Image, ImageDraw
stimuli = []
labels = ["Red Circle", "Blue Circle", "Red Square", "Shifted Circle", "Two Circles", "Pale Circle"]
for i in range(6):
    image = Image.new("RGB", (256, 256), "ivory")
    draw = ImageDraw.Draw(image)
    color = "navy" if i == 1 else ("lightcoral" if i == 5 else "firebrick")
    if i == 2:
        draw.rectangle((65,65,190,190), fill=color)
    elif i == 3:
        draw.ellipse((10,65,135,190), fill=color)
    elif i == 4:
        draw.ellipse((25,85,110,170), fill=color)
        draw.ellipse((145,85,230,170), fill=color)
    else:
        draw.ellipse((65,65,190,190), fill=color)
    stimuli.append(image)
gallery(stimuli, labels, columns=3)
plt.show()

## LPIPS and DreamSim

All distances here use the convention “lower means more similar.” LPIPS compares spatial features after an explicit 256×256 resize and [-1,1] normalization. DreamSim uses its published preprocessing and normalized embeddings. Distances from different metrics have different scales; compare rankings and RDM structure rather than averaging raw scores.

In [ ]:
rdms = {}
for name in ("lpips", "dreamsim"):
    metric = Similarity(name)
    rdms[name] = metric.pairwise(stimuli)
    assert np.allclose(rdms[name], rdms[name].T)
    assert np.allclose(np.diag(rdms[name]), 0)
    assert np.isfinite(rdms[name]).all()
    if name == "dreamsim":
        embeddings = metric.embed(stimuli)
        print("DreamSim embedding shape:", embeddings.shape)
    del metric
    release_memory()

## Text-Prompted Similarity

TPIPS provides a reusable embedding for each image–factor combination. Changing “color” to “shape” changes the representation. Cache keys must therefore include the factor, not just the image. We load the model once and change only the factor between calls.

In [ ]:
metric = Similarity("tpips", factor="color")
rdms["tpips: color"] = metric.pairwise(stimuli)
metric.factor = "shape"
rdms["tpips: shape"] = metric.pairwise(stimuli)
del metric
release_memory()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, distances) in zip(axes, rdms.items()):
    im = ax.imshow(distances, cmap="magma", vmin=0)
    ax.set_title(name)
    ax.set_xticks(range(len(labels)), labels, rotation=90)
    fig.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

## Representational Similarity Analysis

Spearman correlation of the upper-triangular RDM entries tests whether two models order pairs similarly. This is a descriptive model-to-model comparison. Pairwise distances are dependent, so an ordinary correlation p-value across pairs is not an appropriate inferential test. Human validation would require separate participants, conditions, and a suitable resampling or permutation design.

In [ ]:
from scipy.stats import spearmanr
upper = np.triu_indices(len(stimuli), 1)
names = list(rdms)
agreement = np.array([[spearmanr(rdms[a][upper], rdms[b][upper]).statistic for b in names] for a in names])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(agreement, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(names)), names, rotation=45, ha="right")
ax.set_yticks(range(len(names)), names)
fig.colorbar(im, ax=ax, label="Spearman RDM Correlation")
plt.tight_layout()
plt.show()
for name, matrix in rdms.items():
    print(name, "red/blue circle:", round(float(matrix[0,1]), 3), "red circle/square:", round(float(matrix[0,2]), 3))

## Exercise

Does TPIPS reverse the relative ordering of the color-change and shape-change pairs when the factor changes? Report the numerical result even if your prediction is wrong. This small example tests an operational contrast; it does not establish performance on natural images.

In [ ]:
contrasts = {name: float(rdms[name][0,1] - rdms[name][0,2]) for name in ("tpips: color", "tpips: shape")}
print("Positive means the color change is judged more different than the shape change:", contrasts)